# 09 · Real phone video: failure analysis (RQ1), detector on real video (RQ6), Blender export

1. **Video → SMPL without PHALP.** YOLO person detection + a simple IoU tracker + HMR2.0 on the official
   ViTDetDataset crops. This is the fix for the broken notebook 01 environment (Week 1 task).
2. Runs, on the **same crops**: the released HMR2.0 and (if notebooks 07/08 ran) the full
   Diagnose → Specialize → Fuse model. Also converts teammates' outputs (Multi-HMR, MediaPipe, PARE,
   WHAM/GVHMR) into the same standard files.
3. Builds the RQ1 failure table: proxy metrics per clip × model, grouped by the 29-row test plan.
4. RQ6: does the detector trained on Blender images flag the right rows on real video?

Clips: `CS747_project/videos/rows/rowNN_<desc>_<k>.mp4` (the test-plan naming). Results:
`results/real/<clip>__<model>.npz`. Everything is skipped if it already exists.

In [ ]:
TARGET_H, MAX_SECONDS = 720, 60
EXPERTS = ["E0", "E1", "E2", "E3"]
RUN = "experts_r16_b4_s0"                       # notebook 07/08 run (optional)
LEGACY = {"clipA": 9, "clipB": 7}               # clips recorded before the naming scheme -> test-plan row

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/CS747_project"
CODE_DIR    = f"{PROJECT_DIR}/code/rw2b"          # the rw2b folder (pose_lib/, blender/, notebooks/)

from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, time, gc, shutil, subprocess
import numpy as np
if not os.path.exists(f"{CODE_DIR}/pose_lib"):              # first run: unpack the code next to the zip
    !cd "{PROJECT_DIR}/code" && unzip -oq rw2b.zip
sys.path.insert(0, CODE_DIR)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# HMR2.0 without PHALP/detectron2 (the install that worked in notebook 05 on Python 3.13)
%cd /content
![ -d 4D-Humans ] || git clone -q https://github.com/shubham-goel/4D-Humans.git
%cd /content/4D-Humans
!pip install -q numpy yacs smplx pytorch_lightning timm einops omegaconf trimesh webdataset pyrender
!pip install -q --no-build-isolation git+https://github.com/mattloper/chumpy
!pip install -q --no-deps .
!sed -i 's/HMR2.load_from_checkpoint(checkpoint_path, strict=False, cfg=model_cfg)/HMR2.load_from_checkpoint(checkpoint_path, strict=False, cfg=model_cfg, weights_only=False)/' hmr2/models/__init__.py
%cd /content
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
sys.path.insert(0, "/content/4D-Humans")

from hmr2.configs import CACHE_DIR_4DHUMANS
from hmr2.models import download_models
download_models(CACHE_DIR_4DHUMANS)                           # ~2.5 GB, cached for the session

# SMPL: plain-numpy files only, chumpy never imported (the fix for the broken environment)
from pose_lib.smpl import pkl_to_npz, setup_hmr2_smpl
SMPL_NPZ = f"{PROJECT_DIR}/data/smpl_neutral.npz"
if not os.path.exists(SMPL_NPZ):
    print(pkl_to_npz(f"{PROJECT_DIR}/body_models/smpl/basicModel_neutral_lbs_10_207_0_v1.0.0.pkl", SMPL_NPZ))
setup_hmr2_smpl(SMPL_NPZ)

import torch
from pose_lib import hmr2_ft as ft, lora, metrics, stats
fk = ft.SMPLJoints(SMPL_NPZ)
print("HMR2.0 + SMPL ready")

In [ ]:
!pip install -q ultralytics torchvision
from pose_lib import video, convert, experts as ex, detector as dt, fusion, smoothing
from pose_lib.format import PoseSeq, parse_name
from pose_lib.smpl import SMPLJointsNP
fknp = SMPLJointsNP(SMPL_NPZ)
REAL = f"{PROJECT_DIR}/results/real"; os.makedirs(REAL, exist_ok=True)

model, model_cfg = video.load_hmr2_with_cfg(); model.cuda().eval()
predictors = {"hmr2": video.base_predictor(model)}
MODELS = f"{PROJECT_DIR}/models/{RUN}"
if all(os.path.exists(f"{MODELS}/{e}.pt") for e in EXPERTS) and os.path.exists(f"{MODELS}/detector_resnet18.pt"):
    lora.inject(model, lora.hmr2_targets(4), EXPERTS, r=16)
    for e in EXPERTS: lora.load_adapter(model, e, f"{MODELS}/{e}.pt")
    det = dt.Detector("resnet18", pretrained=False).cuda().eval()
    det.load_state_dict(torch.load(f"{MODELS}/detector_resnet18.pt", map_location="cuda"))
    predictors["dgs_weighted"] = ex.DGS(model, det, "weighted")
    predictors["dgs_perjoint"] = ex.DGS(model, det, "per_joint")
    if os.path.exists(f"{MODELS}/learned_fusion.pt"):
        lf = fusion.LearnedFusion(4); lf.load_state_dict(torch.load(f"{MODELS}/learned_fusion.pt"))
        predictors["dgs_learned"] = ex.DGS(model, det, "learned", learned=lf.eval())
model.cuda()
print("predictors:", list(predictors))
yolo = video.load_detector_yolo("yolo11n.pt")

## 1. Run every clip (Week 1 check: start with one clip)

In [ ]:
clips = sorted(glob.glob(f"{PROJECT_DIR}/videos/rows/row*.mp4"))
clips += [p for p in sorted(glob.glob(f"{PROJECT_DIR}/videos/*.mp4")) if os.path.splitext(os.path.basename(p))[0] in LEGACY]
print(len(clips), "clips")
TIMING = f"{REAL}/timing.json"; timing = json.load(open(TIMING)) if os.path.exists(TIMING) else {}
for p in clips:
    clip = os.path.splitext(os.path.basename(p))[0]
    todo = {k: v for k, v in predictors.items() if not os.path.exists(f"{REAL}/{clip}__{k}.npz")}
    if not todo:
        continue
    _, t = video.run(p, REAL, todo, model_cfg, fknp, yolo=yolo, target_h=TARGET_H, max_seconds=MAX_SECONDS)
    timing[clip] = t; json.dump(timing, open(TIMING, "w"), indent=1)

## 2. Teammates' outputs → standard files
Put raw outputs here (any that exist are converted; the rest are skipped):
```
results/real_raw/multihmr/<clip>_smpl.npz           Madeeha (Multi-HMR, same keys as notebook 01)
results/real_raw/mediapipe_<lite|full|heavy>/<clip>_<size>.npz   Nishitha (world, raw, visibility, found)
results/real_raw/pare/<clip>/pare_output.pkl        Gnanitha (PARE demo output)
results/real_raw/gvhmr/<clip>/hmr4d_results.pt      Salma (GVHMR demo output)
results/real_raw/wham/<clip>/wham_output.pkl        Salma (WHAM demo output)
```

In [ ]:
RAW = f"{PROJECT_DIR}/results/real_raw"
import cv2
def clip_info(clip):
    vids = glob.glob(f"{PROJECT_DIR}/videos/rows/{clip}.mp4") + glob.glob(f"{PROJECT_DIR}/videos/{clip}.mp4")
    if not vids: return 30.0, None
    cap = cv2.VideoCapture(vids[0]); fps = cap.get(cv2.CAP_PROP_FPS) or 30; n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    return fps, n
done = []
for f in glob.glob(f"{RAW}/multihmr/*_smpl.npz"):
    clip = os.path.basename(f)[:-9]; out = f"{REAL}/{clip}__multihmr.npz"
    if not os.path.exists(out):
        convert.from_smpl_npz(f, fknp, model="multihmr", clip=clip).save(out); done.append(out)
for f in glob.glob(f"{RAW}/mediapipe_*/*.npz"):
    size = os.path.basename(os.path.dirname(f)).split("_", 1)[1]
    clip = os.path.basename(f)[:-4].rsplit("_", 1)[0]; out = f"{REAL}/{clip}__mediapipe_{size}.npz"
    if not os.path.exists(out) and "world" in np.load(f).files:
        fps, n = clip_info(clip)
        convert.from_mediapipe(f, fps=fps, model=f"mediapipe_{size}", clip=clip, n_frames=n).save(out); done.append(out)
for name, pattern, fn in [("pare", "pare/*/pare_output.pkl", convert.from_pare),
                          ("gvhmr", "gvhmr/*/hmr4d_results.pt", convert.from_gvhmr),
                          ("wham", "wham/*/wham_output.pkl", convert.from_wham)]:
    for f in glob.glob(f"{RAW}/{pattern}"):
        clip = os.path.basename(os.path.dirname(f)); out = f"{REAL}/{clip}__{name}.npz"
        if not os.path.exists(out):
            fps, n = clip_info(clip)
            fn(f, fknp, fps=fps, clip=clip, n_frames=n).save(out); done.append(out)
print(len(done), "converted")

## 3. Temporal baseline: Savitzky–Golay smoothing of every output (`+sg`)

In [ ]:
for f in glob.glob(f"{REAL}/*__*.npz"):
    clip, m, _ = parse_name(f)
    if m.endswith("+sg"): continue
    out = f"{REAL}/{clip}__{m}+sg.npz"
    if not os.path.exists(out):
        s = PoseSeq.load(f); smoothing.smooth_seq(s, fk=fknp if s.is_smpl else None).save(out)

## 4. RQ1 failure table

In [ ]:
import pandas as pd, re
GROUP = {**{r: "People" for r in range(1, 7)}, **{r: "Occlusion" for r in range(7, 12)},
         **{r: "Body position" for r in range(12, 16)}, **{r: "Camera" for r in range(16, 20)},
         **{r: "Video quality" for r in range(20, 25)}}
seqs = {}
for f in sorted(glob.glob(f"{REAL}/*__*.npz")):
    clip, m, row = parse_name(f)
    if re.search(r"_t\d+$", clip):              # secondary tracks (other people) are kept but not tabulated
        continue
    row = row if row > 0 else LEGACY.get(clip, -1)
    seqs[(clip, m)] = (PoseSeq.load(f), row)
rows = []
for (clip, m), (s, row) in seqs.items():
    r = metrics.proxy_report(s); r.update(row=row, group=GROUP.get(row, "?"))
    others = [o for (c2, m2), (o, _) in seqs.items() if c2 == clip and m2 != m and not m2.endswith("+sg")]
    r["disagreement_mm"] = np.nanmean([metrics.agreement(s, o) for o in others]) if others else np.nan
    rows.append(r)
P = pd.DataFrame(rows); P.to_csv(f"{REAL}/proxy_metrics_all.csv", index=False)
cols = ["detection_rate", "jitter_mm", "rot_jitter_deg", "pose_jumps", "lr_swaps", "bone_cv", "foot_skate_mm",
        "low_conf_joints", "disagreement_mm"]
by_group = P.groupby(["group", "model"])[cols].mean().round(3); by_group.to_csv(f"{REAL}/rq1_by_group.csv"); by_group

In [ ]:
# Flags: which failure each model shows on which row (thresholds are deliberately simple; tune once on rows 1/16)
TH = {"detection_rate": ("<", 0.9), "pose_jumps": (">", 0.05), "lr_swaps": (">", 0), "bone_cv": (">", 0.05),
      "jitter_mm": (">", 2 * P[P.row.isin([1, 16])]["jitter_mm"].median() if P.row.isin([1, 16]).any() else 30),
      "disagreement_mm": (">", 80)}
def flags(r):
    out = []
    for k, (op, v) in TH.items():
        x = r[k]
        if np.isfinite(x) and ((op == "<" and x < v) or (op == ">" and x > v)):
            out.append(k)
    return ", ".join(out)
P["flags"] = P.apply(flags, axis=1)
fail = P[~P.model.str.endswith("+sg")].pivot_table(index=["row", "clip"], columns="model", values="flags", aggfunc="first")
fail.to_csv(f"{REAL}/rq1_failure_table.csv"); fail

In [ ]:
# Two-rater visual score sheet (fill in, then merge with the proxy table for the report)
metrics.rating_template(sorted({c for c, _ in seqs}), sorted({m for _, m in seqs if not m.endswith("+sg")}),
                        f"{REAL}/visual_ratings_template.csv")

## 5. RQ6: does the Blender-trained detector flag the right rows on real video?

In [ ]:
def frame_scores(model_name, key):
    xs, ys = [], []
    for (clip, m), (s, row) in seqs.items():
        if m != model_name or f"extra_{key}" not in s.extra: continue
        xs.append(s.extra[f"extra_{key}"]); ys.append(np.full(len(s.extra[f"extra_{key}"]), row))
    return (np.concatenate(xs), np.concatenate(ys)) if xs else (None, None)

POS = {"occlusion": range(7, 12), "quality": range(20, 23), "framing": range(12, 16)}
NEG = [1, 16]
res = {}
src = "dgs_weighted" if any(m == "dgs_weighted" for _, m in seqs) else None
if src:
    cond, row = frame_scores(src, "dgs_cond")          # (frames, 3): occlusion, quality, framing
    for i, name in enumerate(dt.COND_NAMES):
        mask = np.isin(row, list(POS[name]) + NEG)
        res[f"detector AUC {name}"] = dt.auc(np.isin(row[mask], list(POS[name])), cond[mask, i])
for name, key in [("quality", "chk_cond_quality"), ("framing", "chk_cond_framing")]:
    x, row = frame_scores("hmr2", key)
    if x is not None:
        mask = np.isin(row, list(POS[name]) + NEG)
        res[f"cheap-check AUC {name}"] = dt.auc(np.isin(row[mask], list(POS[name])), x[mask])
json.dump(res, open(f"{REAL}/rq6_real_detector_auc.json", "w"), indent=1); res

## 6. Blender: animate a character with any result file
The standard files keep notebook 01's keys, so notebooks 02/03 and `blender/smplx_mocap.py` read them as is.

In [ ]:
CLIP, MODEL = "clipB", "dgs_perjoint+sg"          # pick from results/real
motion = f"{REAL}/{CLIP}__{MODEL}.npz"
!apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 libgl1 libegl1 > /dev/null
![ -d /content/blender ] || (wget -q https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz -C /content && mv /content/blender-4.2.3-linux-x64 /content/blender && rm /content/blender.tar.xz)
OUT = f"{PROJECT_DIR}/outputs/{CLIP}__{MODEL}"
!/content/blender/blender --background --python "{CODE_DIR}/blender/smplx_mocap.py" -- --model "{PROJECT_DIR}/body_models/smplx/SMPLX_NEUTRAL.npz" --motion "{motion}" --blend "{OUT}.blend" --video "{OUT}.mp4" --max-frames 150 2>&1 | grep "smplx_mocap\|Error"